# Freelance Market Monitor — Deep Crawl Scraper
**Platforms:** Freelancer.com · Mostaqel.com  
**Output:** `freelance_data.json`

> Run cells **top-to-bottom**. The scraper uses Selenium + headless Chrome installed directly from Google's `.deb` package (Colab-compatible).

## 1 · Install Chrome & Python Packages

In [ ]:
# Install Google Chrome from Google's .deb (Colab's apt chromium is snap-only and broken)
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
!pip install -q --upgrade selenium requests

import subprocess
r = subprocess.run(['google-chrome', '--version'], capture_output=True, text=True)
print('✅', r.stdout.strip())


Selecting previously unselected package libatk1.0-data.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libatk-bridge2.0-0:amd64.
Preparing to unpack .../03-libatk-bridge2.0-0_2.38.0-3_amd64.deb ...
Unpacking libatk-bridge2.0-0:amd64 (2.38.0-3) ...
Selecting previously unselected package libvulkan1:amd64.
Preparing to unpack .../04-libvulkan1_1.3.204.1-2_amd64.deb ...
Unpacking libvulkan1:amd64 (1.3.204.1-2) ...
Selecting previously unselected package libxcomposite1:amd

## 2 · Scraper Source
The cells below contain the full scraper code, split into logical sections.
`create_driver()` is replaced with a Colab-compatible version that lets Selenium 4.6+ auto-download the matching ChromeDriver — no path configuration needed.

### Imports & Logging

In [ ]:
import json
import logging
import random
import re
import time
from dataclasses import dataclass, field, asdict
import sys
from typing import Optional
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

# ── CS313x Lab Tools & Selenium Migration ───────────────────────────────────
# Kept requests for robots.txt fetching.
# Selenium is introduced to handle modern Javascript rendering & bot bypass,
# and is used exclusively for page fetching and element parsing (no BeautifulSoup).
import requests
import unicodedata

# ── NLTK for Arabic text processing ─────────────────────────────────────────
import nltk
# Download required NLTK data (safe to call repeatedly; skips if already present)
for _pkg in ('stopwords', 'punkt', 'punkt_tab'):
    try:
        nltk.download(_pkg, quiet=True)
    except Exception:
        pass

try:
    from nltk.corpus import stopwords as _sw
    ARABIC_STOPWORDS = set(_sw.words('arabic'))
except Exception:
    ARABIC_STOPWORDS = set()

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
# webdriver_manager not needed — Selenium 4.6+ manages chromedriver automatically

# ---------------------------------------------------------------------------
# Logging Setup
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  [%(levelname)s]  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


### Data Schema

In [ ]:
# ---------------------------------------------------------------------------
# Data Schema
# ---------------------------------------------------------------------------
@dataclass
class FreelanceProject:
    """
    Canonical record for one freelance project.

    DEEP CRAWLING NOTE: description_snippet is now replaced by
    full_description — the complete project description text extracted
    from the individual project detail page, not just a card summary.

    All fields that cannot be found are stored as None (null in JSON).
    """
    platform: str                               # Source platform name
    title: Optional[str] = None                 # Project / job title
    url: Optional[str] = None                   # Direct link to the project
    budget_min: Optional[float] = None          # Minimum budget (numeric)
    budget_max: Optional[float] = None          # Maximum budget (numeric)
    budget_currency: Optional[str] = None       # Currency code, e.g. "USD"
    budget_type: Optional[str] = None           # "fixed" | "hourly" | "unknown"
    skills: list = field(default_factory=list)  # Complete skills list
    category: Optional[str] = None             # Project category / domain
    posted_date: Optional[str] = None          # Raw date string as shown on site
    # ── UPGRADED from description_snippet → full_description ─────────────
    # Surface scraping stored only the ~200-char card teaser.
    # Deep crawling fetches the actual detail page and stores the entire
    # project description body as the professor requires.
    full_description: Optional[str] = None      # Complete description from detail page
    description_snippet: Optional[str] = None   # Card-level teaser (kept as fallback)



### User-Agent Pool & `create_driver` (Colab-Compatible)

In [ ]:
# ---------------------------------------------------------------------------
# CS313x Manual Headers (Lab-Compliant)
# ---------------------------------------------------------------------------
# As taught in Web_Scraping.ipynb:
#   "Websites block bots. Headers make your request look like a browser."
#
# We define a pool of realistic User-Agent strings and rotate them, exactly
# as demonstrated in the lab, to avoid simple bot detection.

USER_AGENTS = [
    # Chrome on Windows — most common desktop browser fingerprint
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    # Safari on macOS
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4_1) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.4.1 Safari/605.1.15",
    # Firefox on Linux
    "Mozilla/5.0 (X11; Linux x86_64; rv:125.0) Gecko/20100101 Firefox/125.0",
    # Edge on Windows
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36 Edg/123.0.0.0",
]



def create_driver():
    """
    Colab-compatible headless Chrome driver.
    Selenium 4.6+ SeleniumManager auto-downloads the matching chromedriver.
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1280,1024")
    # Force UTF-8 charset so Arabic/RTL text is always read correctly
    chrome_options.add_argument("--force-renderer-accessibility")
    chrome_options.add_argument("--lang=ar,en")
    chrome_options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
    prefs = {"profile.managed_default_content_settings.images": 2}
    chrome_options.add_experimental_option("prefs", prefs)
    # No Service() passed — Selenium auto-fetches the matching chromedriver
    driver = webdriver.Chrome(options=chrome_options)
    return driver


### Utility Helpers (Selenium Parsing)

In [ ]:
# ---------------------------------------------------------------------------
# Utility Helpers
# ---------------------------------------------------------------------------

def polite_sleep(min_s: float = 1.5, max_s: float = 4.0) -> None:
    """
    Sleep a random amount of time between min_s and max_s seconds.
    """
    duration = random.uniform(min_s, max_s)
    log.debug("  ↳ sleeping %.2f s …", duration)
    time.sleep(duration)


# ── Native Selenium Parsing Helpers ─────────────────────────────────────────

def find_element_by_selectors(parent, selectors: list[str]):
    """
    Safely find an element using a list of CSS selectors.
    Returns the first matching element, or None if none are found.
    """
    for selector in selectors:
        try:
            return parent.find_element(By.CSS_SELECTOR, selector)
        except NoSuchElementException:
            continue
    return None


def find_elements_by_selectors(parent, selectors: list[str]):
    """
    Safely find elements using a list of CSS selectors.
    Returns a list of matching elements, or an empty list if none are found.
    """
    for selector in selectors:
        try:
            elements = parent.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                return elements
        except NoSuchElementException:
            continue
    return []


def get_text_by_selectors(parent, selectors: list[str]) -> Optional[str]:
    """
    Safely get the text of an element using a list of CSS selectors.
    """
    el = find_element_by_selectors(parent, selectors)
    return el.text.strip() if el else None


def get_attribute_by_selectors(parent, selectors: list[str], attr: str) -> Optional[str]:
    """
    Safely get an attribute of an element using a list of CSS selectors.
    """
    el = find_element_by_selectors(parent, selectors)
    if el:
        val = el.get_attribute(attr)
        return val.strip() if val else None
    return None


def fetch_page_selenium(
    driver,
    url: str,
    retries: int = 3,
    backoff: float = 5.0,
) -> bool:
    """
    Fetch a URL using Selenium, wait for dynamic elements.
    Returns True if successfully loaded, False otherwise.
    """
    for attempt in range(1, retries + 1):
        try:
            log.debug("Fetching listing with Selenium (attempt %d/%d): %s", attempt, retries, url)
            driver.get(url)
            # Give a random polite delay to let scripts run and render
            time.sleep(random.uniform(2.5, 4.5))

            html = driver.page_source
            if html and len(html) > 200:
                return True

        except Exception as exc:
            log.warning("Attempt %d failed to fetch %s via Selenium: %s", attempt, url, exc)

        if attempt < retries:
            polite_sleep(backoff, backoff * 2)

    log.error("All %d Selenium fetch attempts failed for: %s", retries, url)
    return False


def fetch_detail_page_selenium(driver, url: str) -> bool:
    """
    Fetch a single detail page using Selenium (single attempt, polite rendering time).
    Returns True if successfully loaded, False otherwise.
    """
    try:
        log.debug("Fetching detail page with Selenium: %s", url)
        driver.get(url)
        time.sleep(random.uniform(2.0, 3.5))
        html = driver.page_source
        if html and len(html) > 200:
            return True
    except Exception as exc:
        log.debug("Failed to fetch detail page %s: %s", url, exc)
    return False



### robots.txt Compliance

In [ ]:
# ---------------------------------------------------------------------------
# robots.txt Compliance Helper
# ---------------------------------------------------------------------------

def _check_wildcard_disallow(robots_text: str, path: str) -> bool:
    """
    Python's RobotFileParser ignores the '*' wildcard in Disallow rules.
    This helper manually checks whether any wildcard Disallow pattern
    (e.g. 'Disallow: /search*') matches the given path.

    Returns True if a wildcard rule BLOCKS the path, False otherwise.
    """
    in_wildcard_section = False
    for raw_line in robots_text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        lower = line.lower()
        if lower.startswith("user-agent:"):
            agent = lower.split(":", 1)[1].strip()
            in_wildcard_section = (agent == "*")
        elif in_wildcard_section and lower.startswith("disallow:"):
            rule = line.split(":", 1)[1].strip()
            if "*" in rule:
                prefix = rule.split("*")[0]
                if path.startswith(prefix):
                    return True
    return False


def is_allowed_by_robots(base_url: str, path: str = "/") -> bool:
    """
    Check whether the given path is allowed by the site's robots.txt.

    Uses a standard requests.get() call (lab-compliant) to fetch the
    robots.txt file so we can inspect the HTTP status code before parsing.

    Per RFC 9309:
      200  → parse and obey
      404 / 410  → no rules → allow
      401 / 403  → treat as unavailable → allow (fail open)
      5xx  → temporary error → allow (fail open)
    """
    robots_url = urljoin(base_url, "/robots.txt")
    target_url = urljoin(base_url, path)

    try:
        resp = requests.get(
            robots_url,
            timeout=10,
            headers={"User-Agent": random.choice(USER_AGENTS)},
        )

        if resp.status_code == 200:
            rp = RobotFileParser()
            rp.set_url(robots_url)
            rp.parse(resp.text.splitlines())

            stdlib_allowed = rp.can_fetch("*", target_url)
            wildcard_blocked = _check_wildcard_disallow(resp.text, path)
            allowed = stdlib_allowed and not wildcard_blocked

            if not allowed:
                log.warning(
                    "robots.txt explicitly disallows: %s  (rule blocks %s)",
                    target_url, path,
                )
            else:
                log.info("robots.txt allows: %s", target_url)
            return allowed

        elif resp.status_code in (404, 410):
            log.info(
                "robots.txt not found (HTTP %d) for %s → assuming allowed.",
                resp.status_code, base_url,
            )
            return True

        elif resp.status_code in (401, 403):
            log.info(
                "robots.txt returned HTTP %d for %s → treating as allowed.",
                resp.status_code, base_url,
            )
            return True

        elif resp.status_code >= 500:
            log.warning(
                "robots.txt server error HTTP %d for %s → failing open.",
                resp.status_code, base_url,
            )
            return True

        else:
            log.warning(
                "Unexpected HTTP %d fetching robots.txt for %s → allowing.",
                resp.status_code, base_url,
            )
            return True

    except requests.exceptions.RequestException as exc:
        log.warning("Could not reach robots.txt at %s: %s → allowing.", robots_url, exc)
        return True



### Budget Parser

In [ ]:
# ---------------------------------------------------------------------------
# Arabic Text Utilities
# ---------------------------------------------------------------------------

def normalize_arabic(text: Optional[str]) -> Optional[str]:
    """
    Normalise Arabic text extracted from the page:
      - Strip RTL/LTR marks and zero-width characters
      - Normalise Unicode (NFC)
      - Collapse duplicate whitespace
    Returns None if the result is empty.
    """
    if not text:
        return None
    # Remove directional and zero-width control chars
    text = re.sub(r"[\u200b-\u200f\u202a-\u202e\u2066-\u2069\ufeff]", "", text)
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def clean_arabic_skills(raw_skills: list) -> list:
    """
    Deduplicate and normalise a list of Arabic (or mixed) skill strings.
    Removes NLTK Arabic stopwords so stray words aren't treated as skills.
    """
    seen = set()
    cleaned = []
    for s in raw_skills:
        s = normalize_arabic(s) or ""
        if not s or s.lower() in ARABIC_STOPWORDS:
            continue
        key = s.strip()
        if key not in seen:
            seen.add(key)
            cleaned.append(key)
    return cleaned


def extract_arabic_budget(text: Optional[str]) -> Optional[str]:
    """
    Mostaql shows budgets in Arabic numerals and mixed currency labels.
    Convert Eastern Arabic-Indic digits (٠١٢…٩) to Western (0-9) so that
    clean_budget() can parse the numbers correctly.
    """
    if not text:
        return None
    # Eastern Arabic-Indic → Western digits
    tr = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
    return text.translate(tr)


# ---------------------------------------------------------------------------
# Budget Parser
# ---------------------------------------------------------------------------

def clean_budget(raw: Optional[str]):
    """
    Parse a messy budget string such as:
        "$50 - $100"   →  min=50.0, max=100.0, currency="USD"
        "£500"         →  min=500.0, max=500.0, currency="GBP"
        "SR 200 - 500" →  min=200.0, max=500.0, currency="SAR"
        "Negotiable"   →  min=None,  max=None,  currency=None

    Returns a tuple: (min_val, max_val, currency, budget_type)
    """
    if not raw:
        return None, None, None, "unknown"

    raw = extract_arabic_budget(raw)  # convert Arabic-Indic digits first
    if not raw:
        return None, None, None, "unknown"
    raw = raw.strip()

    currency_map = {
        "$": "USD", "£": "GBP", "€": "EUR",
        "SAR": "SAR", "SR": "SAR", "ر.س": "SAR",
        "EGP": "EGP", "ج.م": "EGP",
    }
    currency = None
    for symbol, code in currency_map.items():
        if symbol in raw:
            currency = code
            break

    budget_type = "hourly" if "/hr" in raw.lower() or "hour" in raw.lower() else "fixed"

    numbers = re.findall(r"[\d,]+\.?\d*", raw.replace(",", ""))
    nums = [float(n) for n in numbers if n]

    if len(nums) == 0:
        return None, None, currency, "unknown"
    elif len(nums) == 1:
        return nums[0], nums[0], currency, budget_type
    else:
        return min(nums), max(nums), currency, budget_type



### Scraper 1 — Freelancer.com

In [ ]:
# ---------------------------------------------------------------------------
# Scraper 1: Freelancer.com
# ---------------------------------------------------------------------------

FREELANCER_BASE   = "https://www.freelancer.com"
FREELANCER_SEARCH = "/jobs/"


def scrape_freelancer(
    driver,
    max_pages: int = 10,
    category_slug: str = "",
) -> list[FreelanceProject]:
    """
    Scrape project listings from Freelancer.com using DEEP CRAWLING and Selenium only.
    """
    projects: list[FreelanceProject] = []
    search_path = FREELANCER_SEARCH + category_slug
    seen_urls: set[str] = set()

    if not is_allowed_by_robots(FREELANCER_BASE, search_path):
        log.warning("Freelancer.com robots.txt blocks this path. Skipping.")
        return projects

    log.info("▶ Starting Freelancer.com DEEP CRAWL (max %d pages) …", max_pages)

    for page_num in range(1, max_pages + 1):
        page_url = f"{FREELANCER_BASE}{search_path}?page={page_num}"
        log.info("  [Listing] Page %d/%d → %s", page_num, max_pages, page_url)

        success = fetch_page_selenium(driver, page_url)
        if not success:
            log.warning("  Could not fetch listing page %d. Stopping.", page_num)
            break

        cards = find_elements_by_selectors(driver, [
            "div.JobSearchCard-item",
            "div[class*='job-card']",
            "li.job-wrap",
            "div.search-result-item"
        ])

        if not cards:
            log.warning(
                "  No job cards on page %d. Site layout may have changed.", page_num
            )
            break

        log.info("  Found %d project cards on page %d.", len(cards), page_num)

        # Extract card-level info from all cards first to prevent stale references when navigating
        extracted_cards = []
        for card in cards:
            card_info = _parse_freelancer_card_selenium(card)
            if card_info:
                extracted_cards.append(card_info)

        for card_idx, card_info in enumerate(extracted_cards, start=1):
            project_url = card_info["url"]

            if not project_url:
                log.debug("    Card %d: no URL found, skipping detail fetch.", card_idx)
                bmin, bmax, currency, btype = clean_budget(card_info["raw_budget_card"])
                project = FreelanceProject(
                    platform="Freelancer.com",
                    title=card_info["title"],
                    url=None,
                    budget_min=bmin,
                    budget_max=bmax,
                    budget_currency=currency,
                    budget_type=btype,
                    skills=card_info["skills_card"],
                    category=card_info["category"],
                    posted_date=card_info["posted"],
                    full_description=None,
                    description_snippet=card_info["snippet"],
                )
                projects.append(project)
                continue

            if project_url in seen_urls:
                log.debug("    Card %d: duplicate URL skipped: %s", card_idx, project_url)
                continue
            seen_urls.add(project_url)

            log.debug(
                "    [Deep Crawl] Card %d/%d — visiting detail page: %s",
                card_idx, len(extracted_cards), project_url,
            )
            polite_sleep(2, 5)

            detail_success = fetch_detail_page_selenium(driver, project_url)
            if detail_success:
                detail_data = _parse_freelancer_detail_selenium(driver)
            else:
                detail_data = {"full_description": None, "skills": [], "budget_raw": None}

            skills_final = detail_data["skills"] if detail_data["skills"] else card_info["skills_card"]
            raw_budget_final = detail_data["budget_raw"] or card_info["raw_budget_card"]
            bmin, bmax, currency, btype = clean_budget(raw_budget_final)

            project = FreelanceProject(
                platform="Freelancer.com",
                title=card_info["title"],
                url=project_url,
                budget_min=bmin,
                budget_max=bmax,
                budget_currency=currency,
                budget_type=btype,
                skills=skills_final,
                category=card_info["category"],
                posted_date=card_info["posted"],
                full_description=detail_data["full_description"],
                description_snippet=card_info["snippet"],
            )
            projects.append(project)
            log.debug(
                "    ✔ Card %d — title: %s | skills: %d | desc_len: %d",
                card_idx,
                (project.title or "")[:50],
                len(project.skills),
                len(project.full_description or ""),
            )

        log.info("  → %d projects collected so far.", len(projects))
        polite_sleep()

    log.info("✔ Freelancer.com DEEP CRAWL done. Total: %d projects.", len(projects))
    return projects


def _parse_freelancer_card_selenium(card) -> Optional[dict]:
    try:
        title = get_text_by_selectors(card, [
            "a.JobSearchCard-primary-heading-link",
            "h2.JobSearchCard-primary-heading a",
            "[class*='heading'] a"
        ])
        if not title:
            return None

        url = get_attribute_by_selectors(card, [
            "a.JobSearchCard-primary-heading-link",
            "h2.JobSearchCard-primary-heading a",
            "[class*='heading'] a",
            "a[href*='/projects/']"
        ], "href")

        raw_budget_card = get_text_by_selectors(card, [
            "div.JobSearchCard-primary-price",
            "[class*='price']",
            "[class*='budget']"
        ])

        skills_elements = find_elements_by_selectors(card, [
            "a.JobSearchCard-primary-tagsLink",
            "[class*='skill'] a",
            "[class*='tag'] a"
        ])
        skills_card = [el.text.strip() for el in skills_elements if el.text.strip()]

        category = get_text_by_selectors(card, [
            "a.JobSearchCard-primary-category",
            "[class*='category']"
        ])

        snippet = get_text_by_selectors(card, [
            "p.JobSearchCard-secondary-description",
            "[class*='description']"
        ])
        if snippet:
            snippet = snippet[:250]

        posted = get_text_by_selectors(card, [
            "span[class*='ago']",
            "time"
        ])

        return {
            "title": title,
            "url": url,
            "raw_budget_card": raw_budget_card,
            "skills_card": skills_card,
            "category": category,
            "snippet": snippet,
            "posted": posted
        }
    except Exception as exc:
        log.warning("  Error parsing Freelancer card: %s", exc)
        return None


def _parse_freelancer_detail_selenium(driver) -> dict:
    result = {"full_description": None, "skills": [], "budget_raw": None}

    desc_element = find_element_by_selectors(driver, [
        "p.Project-description",
        "div.PageProjectViewLogout-projectDescription",
        "div.project-description",
        "[class*='ProjectDescription']",
        "[class*='project-description']",
        "div[class*='description'] p",
        "section.project-description"
    ])
    if desc_element:
        result["full_description"] = desc_element.text.strip()

    skills_elements = find_elements_by_selectors(driver, [
        "a[href*='/jobs/']",
        "a.skill-tag",
        "[class*='SkillTag']",
        "[class*='skill-tag']",
        "ul.skills-list li"
    ])
    result["skills"] = [el.text.strip() for el in skills_elements if el.text.strip()]

    budget_element = find_element_by_selectors(driver, [
        "h2.text-right",
        "h2.text-body-24",
        "[class*='PageProjectViewLogout-budget']",
        "[class*='project-budget']",
        "[class*='Budget']",
        "span[class*='price']"
    ])
    if budget_element:
        result["budget_raw"] = budget_element.text.strip()

    return result



### Scraper 2 — Mostaqel.com

In [ ]:
# ---------------------------------------------------------------------------
# Scraper 2: Mostaqel.com
# ---------------------------------------------------------------------------

MOSTAQEL_BASE     = "https://mostaql.com"
MOSTAQEL_PROJECTS = "/projects"


def scrape_mostaqel(
    driver,
    max_pages: int = 10,
) -> list[FreelanceProject]:
    """
    Scrape project listings from Mostaql.com (مستقل) using DEEP CRAWLING and Selenium only.
    """
    projects: list[FreelanceProject] = []
    seen_urls: set[str] = set()

    if not is_allowed_by_robots(MOSTAQEL_BASE, MOSTAQEL_PROJECTS):
        log.warning("Mostaqel robots.txt blocks project listings. Skipping.")
        return projects

    log.info("▶ Starting Mostaqel.com DEEP CRAWL (max %d pages) …", max_pages)

    for page_num in range(1, max_pages + 1):
        page_url = f"{MOSTAQEL_BASE}{MOSTAQEL_PROJECTS}?page={page_num}"
        log.info("  [Listing] Page %d/%d → %s", page_num, max_pages, page_url)

        success = fetch_page_selenium(driver, page_url)
        if not success:
            log.warning("  Failed to fetch listing page %d. Stopping.", page_num)
            break

        cards = find_elements_by_selectors(driver, [
            "table.projects-table tbody tr",
            "div.project-row",
            "[class*='project-card']",
            "article.project"
        ])

        if not cards:
            log.warning("  No job cards found on page %d.", page_num)
            break

        log.info("  Found %d project cards on page %d.", len(cards), page_num)

        # Extract card-level info from all cards first to prevent stale references when navigating
        extracted_cards = []
        for card in cards:
            card_info = _parse_mostaqel_card_selenium(card)
            if card_info:
                extracted_cards.append(card_info)

        for card_idx, card_info in enumerate(extracted_cards, start=1):
            project_url = card_info["url"]

            if not project_url:
                log.debug("    Card %d: no URL found, skipping detail fetch.", card_idx)
                bmin, bmax, currency, btype = clean_budget(card_info["raw_budget_card"])
                project = FreelanceProject(
                    platform="Mostaqel.com",
                    title=card_info["title"],
                    url=None,
                    budget_min=bmin,
                    budget_max=bmax,
                    budget_currency=currency,
                    budget_type=btype,
                    skills=card_info["skills_card"],
                    category=card_info["category"],
                    posted_date=card_info["posted"],
                    full_description=None,
                    description_snippet=card_info["snippet"],
                )
                projects.append(project)
                continue

            if project_url in seen_urls:
                log.debug("    Card %d: duplicate URL skipped: %s", card_idx, project_url)
                continue
            seen_urls.add(project_url)

            log.debug(
                "    [Deep Crawl] Card %d/%d — visiting detail page: %s",
                card_idx, len(extracted_cards), project_url,
            )
            polite_sleep(2, 5)

            detail_success = fetch_detail_page_selenium(driver, project_url)
            if detail_success:
                detail_data = _parse_mostaqel_detail_selenium(driver)
            else:
                detail_data = {"full_description": None, "skills": [], "budget_raw": None}

            skills_final = detail_data["skills"] if detail_data["skills"] else card_info["skills_card"]
            raw_budget_final = detail_data["budget_raw"] or card_info["raw_budget_card"]
            bmin, bmax, currency, btype = clean_budget(raw_budget_final)

            project = FreelanceProject(
                platform="Mostaqel.com",
                title=card_info["title"],
                url=project_url,
                budget_min=bmin,
                budget_max=bmax,
                budget_currency=currency,
                budget_type=btype,
                skills=skills_final,
                category=card_info["category"],
                posted_date=card_info["posted"],
                full_description=detail_data["full_description"],
                description_snippet=card_info["snippet"],
            )
            projects.append(project)
            log.debug(
                "    ✔ Card %d — title: %s | skills: %d | desc_len: %d",
                card_idx,
                (project.title or "")[:50],
                len(project.skills),
                len(project.full_description or ""),
            )

        log.info("  → %d projects collected so far.", len(projects))
        polite_sleep()

    log.info("✔ Mostaqel.com DEEP CRAWL done. Total: %d projects.", len(projects))
    return projects



# Arabic month names — used to reject date strings in the budget fallback
_ARABIC_MONTHS = {
    'يناير','فبراير','مارس','أبريل','مايو','يونيو',
    'يوليو','أغسطس','سبتمبر','أكتوبر','نوفمبر','ديسمبر',
}


def _looks_like_budget(text: str) -> bool:
    """
    Return True only if text genuinely looks like a price/budget string.

    Rejects:
      - Date strings like "28 مارس 2023"  (contain Arabic month names)
      - Percentages like "100.00%"         (hire-rate column)
      - Plain counts like "1", "0"         (open-projects column)
      - Long strings (descriptions)
    Accepts:
      - "$25.00 - $50.00", "€ 100", "SAR 500 - 1000", "250 ريال", etc.
    """
    if not text or len(text) > 60:
        return False
    # Must contain at least one real currency symbol OR an explicit SAR label
    has_currency = bool(re.search(r'[$€£]', text)) or bool(
        re.search(r'\b(SAR|SR|ريال|ر\.س)\b', text)
    )
    if not has_currency:
        return False
    # Must contain a digit
    if not re.search(r'\d', text):
        return False
    # Reject date strings — any Arabic month name disqualifies the cell
    if set(text.split()) & _ARABIC_MONTHS:
        return False
    # Reject bare percentages (hire-rate column)
    if re.fullmatch(r'[\d,\.]+%', text.strip()):
        return False
    return True


def _find_budget_by_label(driver_or_card) -> Optional[str]:
    """
    Mostaql renders its project info as a label→value table:
        <th>الميزانية</th><td>$25.00 - $50.00</td>

    Strategy (in order):
      1. XPath: find the <td>/<dd> that DIRECTLY follows the Arabic label "الميزانية"
         — this is the most reliable and specific match.
      2. Class-name fallback: legacy selectors in case Mostaql restructures.
      3. Full-page TD scan: last resort, guarded by _looks_like_budget() so we
         never accidentally pick up dates, hire-rates, or project counts.
    """
    from selenium.webdriver.common.by import By

    # ── 1. Primary: label-relative XPath ─────────────────────────────────────
    for xpath in [
        "//th[contains(., 'الميزانية')]/following-sibling::td[1]",
        "//dt[contains(., 'الميزانية')]/following-sibling::dd[1]",
        "//td[contains(., 'الميزانية')]/following-sibling::td[1]",
        "//li[contains(., 'الميزانية')]//span[last()]",
    ]:
        try:
            els = driver_or_card.find_elements(By.XPATH, xpath)
            for el in els:
                text = el.text.strip()
                if _looks_like_budget(text):
                    return text
        except Exception:
            continue

    # ── 2. Class-name fallback ────────────────────────────────────────────────
    for xpath in [
        "//*[contains(@class,'budget')]",
        "//*[contains(@class,'price')]",
    ]:
        try:
            els = driver_or_card.find_elements(By.XPATH, xpath)
            for el in els:
                text = el.text.strip()
                if _looks_like_budget(text):
                    return text
        except Exception:
            continue

    # ── 3. Last-resort full-page TD scan ─────────────────────────────────────
    # Guarded by _looks_like_budget(): rejects dates, percentages, counts, etc.
    for tag in ("td", "dd", "span"):
        try:
            for el in driver_or_card.find_elements(By.TAG_NAME, tag):
                text = el.text.strip()
                if _looks_like_budget(text):
                    return text
        except Exception:
            continue

    return None

def _parse_mostaqel_card_selenium(card) -> Optional[dict]:
    try:
        title = normalize_arabic(get_text_by_selectors(card, [
            "h2.project__title a",
            "h2 a",
            "a.project-title",
            "[class*='title'] a",
            "td.title-cell a"
        ]))
        if not title:
            return None

        url = get_attribute_by_selectors(card, [
            "h2.project__title a",
            "h2 a",
            "a.project-title",
            "[class*='title'] a",
            "a[href*='/projects/']"
        ], "href")

        raw_budget_card = extract_arabic_budget(_find_budget_by_label(card))

        skills_elements = find_elements_by_selectors(card, [
            "ul.project__skills li",
            "ul.skills-list li",
            "[class*='skill']",
            "span.tag",
            "a.tag"
        ])
        # Normalise Arabic skill names and strip stopwords
        skills_card = clean_arabic_skills(
            [el.text.strip() for el in skills_elements if el.text.strip()]
        )

        category = normalize_arabic(get_text_by_selectors(card, [
            "a.project__category",
            "[class*='category'] a",
            "span.category",
            "td.category-cell a"
        ]))

        snippet = normalize_arabic(get_text_by_selectors(card, [
            "div.project__brief",
            "p.project-description",
            "[class*='description']",
            "div.carda__content p"
        ]))
        if snippet:
            snippet = snippet[:250]

        date_element = find_element_by_selectors(card, ["time", "[class*='date']"])
        posted = None
        if date_element:
            posted = date_element.get_attribute("datetime")
            if not posted:
                posted = date_element.text.strip()

        return {
            "title": title,
            "url": url,
            "raw_budget_card": raw_budget_card,
            "skills_card": skills_card,
            "category": category,
            "snippet": snippet,
            "posted": posted
        }
    except Exception as exc:
        log.warning("  Error parsing Mostaqel card: %s", exc)
        return None


def _parse_mostaqel_detail_selenium(driver) -> dict:
    result = {"full_description": None, "skills": [], "budget_raw": None}

    desc_element = find_element_by_selectors(driver, [
        "div.project__brief--full",
        "div.project-details__description",
        "[class*='project__description']",
        "[class*='ProjectDescription']",
        "div.carda__content p",
        "section.project-description",
        "[itemprop='description']"
    ])
    if desc_element:
        # normalize_arabic handles RTL marks, zero-width chars, Unicode normalisation
        result["full_description"] = normalize_arabic(desc_element.text.strip())

    skills_elements = find_elements_by_selectors(driver, [
        "ul.project__skills li",
        "ul.skills-list li",
        "[class*='skill-tag']",
        "[class*='SkillsList'] li",
        "span.tag",
        "a.tag",
        "a[href*='/projects?skill=']"
    ])
    # Deduplicate + remove Arabic stopwords from skill tokens
    result["skills"] = clean_arabic_skills(
        [el.text.strip() for el in skills_elements if el.text.strip()]
    )

    # Use label-based XPath lookup — Mostaql has no reliable budget CSS class
    raw_budget = _find_budget_by_label(driver)
    if raw_budget:
        result["budget_raw"] = extract_arabic_budget(raw_budget)

    return result



### JSON Exporter & Main Orchestrator

In [ ]:
# ---------------------------------------------------------------------------
# JSON Exporter
# ---------------------------------------------------------------------------

def export_to_json(projects: list[FreelanceProject], filepath: str = "freelance_data.json") -> None:
    """
    Serialise the list of FreelanceProject dataclasses to a well-structured
    JSON file.

    DEEP CRAWLING NOTE: Each record now contains full_description (from the
    detail page) in addition to description_snippet (card-level teaser).

    Schema per record:
    {
        "platform":             "Freelancer.com",
        "title":                "Build a REST API",
        "url":                  "https://www.freelancer.com/projects/...",
        "budget_min":           50.0,
        "budget_max":           150.0,
        "budget_currency":      "USD",
        "budget_type":          "fixed",
        "skills":               ["Python", "Django", "REST API"],
        "category":             "Web Development",
        "posted_date":          "2 hours ago",
        "full_description":     "We are looking for an experienced developer …
                                  (full body text from detail page)",
        "description_snippet":  "Looking for an experienced developer …"
    }
    """
    output = {
        "metadata": {
            "total_records": len(projects),
            "platforms": list({p.platform for p in projects}),
            "scraped_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "schema_version": "2.0",          # bumped: now includes full_description
            "crawl_type": "deep",             # documents that this is Deep Crawl data
        },
        "projects": [asdict(p) for p in projects],
    }

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    log.info("💾 Saved %d records → %s", len(projects), filepath)


# ---------------------------------------------------------------------------
# Main Entry Point
# ---------------------------------------------------------------------------

def main():
    """
    Orchestrates the full Deep Crawl ETL pipeline using Selenium only:
      1. Create a headless Chrome Selenium WebDriver.
      2. DEEP CRAWL each platform:
           a. Fetch listing pages (pagination) using Selenium.
           b. Extract individual project URLs from cards.
           c. Visit each project's detail page using Selenium.
           d. Parse full description + complete skills from detail page natively with Selenium.
      3. Merge results.
      4. Export to JSON.
      5. Ensure the Selenium driver is safely quit.
    """
    log.info("=" * 60)
    log.info("  Freelance Market Monitor — DEEP CRAWL Scraper Starting")
    log.info("  Crawl type  : Deep Crawling (following links)")
    log.info("  HTTP client : Selenium Chrome WebDriver (Headless)")
    log.info("  Parser      : Selenium (Native)")
    log.info("  Delay       : time.sleep()  [CS313x compliant]")
    log.info("  Note        : polite_sleep(2,5) between EVERY project visit")
    log.info("=" * 60)

    max_pages = 1000 # Increased to 1000 to collect the full available dataset
    if "--test" in sys.argv:
        max_pages = 1
        log.info("🧪 Running in TEST mode: limiting crawl to 1 page per platform.")

    driver = create_driver()
    all_projects: list[FreelanceProject] = []
    freelancer_projects = []
    mostaqel_projects = []

    try:
        # ── Platform 1: Freelancer.com (Deep Crawl) ───────────────────────────
        freelancer_projects = scrape_freelancer(driver, max_pages=max_pages)
        all_projects.extend(freelancer_projects)

        # Brief pause between platforms
        polite_sleep(3, 7)

        # ── Platform 2: Mostaqel.com (Deep Crawl) ─────────────────────────────
        mostaqel_projects = scrape_mostaqel(driver, max_pages=max_pages)
        all_projects.extend(mostaqel_projects)

    finally:
        log.info("Closing Selenium WebDriver...")
        driver.quit()

    # ── Summary ────────────────────────────────────────────────────────────
    log.info("=" * 60)
    log.info("  DEEP CRAWL COMPLETE")
    log.info("  Freelancer.com : %d projects", len(freelancer_projects))
    log.info("  Mostaqel.com   : %d projects", len(mostaqel_projects))
    log.info("  TOTAL          : %d projects", len(all_projects))
    log.info("=" * 60)

    if not all_projects:
        log.warning("No data collected. The sites' HTML structure may have changed.")
        log.warning("Run with DEBUG logging: logging.basicConfig(level=logging.DEBUG)")
        return

    export_to_json(all_projects, "freelance_data.json")


    main()



## 3 · Run — Test Mode (1 page per platform)
Verify everything works before the full crawl.

In [ ]:
import sys
sys.argv = ['scraper', '--test']   # limits to 1 page per platform

main()


## 4 · Run — Full Crawl
Crawls up to 1 000 pages per platform. This may take a while ☕

In [ ]:
import sys
sys.argv = ['scraper']   # no --test flag

main()


## 5 · Inspect Results & Download

In [ ]:
import json

with open('freelance_data.json', encoding='utf-8') as f:
    data = json.load(f)

meta = data['metadata']
print('=== Crawl Summary ===')
print(f"  Total records : {meta['total_records']}")
print(f"  Platforms     : {meta['platforms']}")
print(f"  Scraped at    : {meta['scraped_at']}")
print()

for p in data['projects'][:3]:
    print(f"  [{p['platform']}] {p['title']}")
    print(f"    URL    : {p['url']}")
    print(f"    Budget : {p['budget_min']} – {p['budget_max']} {p['budget_currency']}")
    print(f"    Skills : {p['skills'][:5]}")
    print()


In [ ]:
from google.colab import files
files.download('freelance_data.json')
